# NexRoute | Delivery Performance Analysis
**Author:** John Paul Torres  
**Scenario:** Fictional logistics operator; January 2024 to December 2025.

**Questions:** Where is service deteriorating, which couriers need review, what does delivery cost, and which geographic corridors matter? Calendar-day SLAs are based on dispatch date. All monetary values are simulated PHP. Run from the repository root.

In [ ]:
from pathlib import Path
import sqlite3, sys
ROOT=Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.database import read_shipments, read_table, DB_PATH
from src.data_processing import prepare_shipments
from src.delivery_metrics import overview, grouped_performance, insights
from src.sla_analysis import sla_summary, pareto
from src.geospatial_analysis import destination_points, geodesic_km
shipments=prepare_shipments(read_shipments())
shipments.shape

## 1. Dataset inspection and quality
Checks include unique keys, nullable completion dates, categorical status frequencies, and reference counts. Cancelled and unresolved cases are not failed deliveries.

In [ ]:
print(shipments.Delivery_Status.value_counts(dropna=False))
assert shipments.Shipment_ID.is_unique
assert shipments.Dispatch_Date.ge(shipments.Shipment_Date).all()
assert shipments.loc[shipments.Delivered,"Actual_Delivery_Date"].notna().all()
assert shipments.loc[~shipments.Delivered,"Actual_Delivery_Date"].isna().all()
shipments.isna().sum().sort_values(ascending=False).head(12)

## 2. Database joins and relationships
SQLite includes normalized courier, hub and SLA dimensions. A joined query below verifies that the applicable SLA matches each origin–destination pair.

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    joined=pd.read_sql_query("""SELECT s.Shipment_ID,s.Service_Type,h.Region AS Origin_Region,
    s.Destination_Region,r.SLA_Days FROM shipments s JOIN hubs h ON s.Origin_Hub_ID=h.Hub_ID
    JOIN sla_rules r ON s.SLA_ID=r.SLA_ID LIMIT 10""",con)
joined

## 3. Delivery KPIs
Success = delivered / (delivered + failed + returned). On-time = delivered on or before the SLA date / delivered. In transit and cancelled are excluded from resolved-rate denominators.

In [ ]:
overview(shipments)

## 4. Volume and seasonal patterns

In [ ]:
monthly=shipments.groupby("Month").size().rename("Shipments").reset_index()
fig=px.line(monthly,x="Month",y="Shipments",title="Synthetic monthly shipment volume")
fig.show()

## 5. Courier performance
Restrict scorecards to couriers with sufficient completed volume before ranking.

In [ ]:
courier=grouped_performance(shipments,["Courier_ID","Courier_Name","Assigned_Region","Vehicle_Type"])
courier[courier.delivered>=75].nlargest(12,"on_time_rate")[["Courier_Name","delivered","on_time_rate","success_rate","avg_days"]]

## 6. SLA compliance and delivery-time distributions

In [ ]:
sla_summary(shipments)

In [ ]:
fig,ax=plt.subplots(figsize=(9,3.5))
sns.histplot(shipments.loc[shipments.Delivered,"Delivery_Days"],bins=20,ax=ax,color="steelblue")
ax.set(title="Delivery time (calendar days)",xlabel="Days from dispatch to delivery")
plt.show()

In [ ]:
pareto(shipments).head(10)

## 7. Geographic analysis
Geometries represent destination city centers, not street addresses. Coordinates are WGS84 and do not represent road distances.

In [ ]:
geo=destination_points(shipments)
print(geo.crs,geo.shape)
geo[["Destination_Region","Destination_City","shipments","on_time_rate","geometry"]].head()

In [ ]:
fig=px.scatter_geo(geo,lat="Destination_Latitude",lon="Destination_Longitude",size="shipments",color="on_time_rate",
    hover_name="Destination_City",scope="asia",projection="mercator",title="Destination volume and SLA")
fig.update_geos(lataxis_range=[4,21],lonaxis_range=[116,127])
fig.show()

## 8. Cost, distance and weight correlations
The simulated cost function explicitly depends on modeled route distance, weight, service and vehicle. These correlations are expected by construction and must not be treated as empirical causal findings.

In [ ]:
cols=["Distance_KM","Package_Weight_KG","Shipping_Cost_PHP","Delivery_Days"]
shipments[cols].corr(numeric_only=True).round(3)

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
sns.heatmap(shipments[cols].corr(),annot=True,cmap="coolwarm",center=0,ax=ax)
plt.show()

## 9. Region-level priorities and findings
Insights below are recomputed, rather than hardcoded in notebook prose.

In [ ]:
print("\n".join(insights(shipments)))
by_region=grouped_performance(shipments,"Destination_Region")
by_region.sort_values("on_time_rate")[["Destination_Region","shipments","success_rate","on_time_rate","avg_cost"]]

## 10. Operational recommendations
Consider capacity reviews in the lowest on-time region; review route exceptions and incorrect-address cases; compare courier performance only after volume thresholds. These are hypotheses for investigation, not measured improvement claims.

### Limitations
All shipments, couriers, hubs and costs are fictional. City-center points are representative, not street geocodes. Straight-line distances are not driving distances. No official province polygons, actual traffic data, weather feeds, carrier billing, or real SLA contracts were used. Unresolved records are evaluated as of 2025-12-31.